- identify env parameters from data
- estimate the parameter uncertainty
    - Do multiple runs and consider the variance?
- **probably makes sense to build this as parallel as possible compared to the model training**

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"]="false"
from functools import partial
import time
from copy import deepcopy
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
import jax
import jax.numpy as jnp
# jax.config.update("jax_enable_x64", True)
gpus = jax.devices()
print(gpus)

jax.config.update("jax_default_device", gpus[0])

import diffrax
import equinox as eqx
import optax

from haiku import PRNGSequence

import exciting_environments as excenvs

from dmpe.data_management import DataPaths
from dmpe.models.models import NeuralEulerODECartpole
from dmpe.evaluation.experiment_utils import get_experiment_ids, load_experiment_results, load_all_experiment_results
from dmpe.evaluation.data_evaluation import DataEvaluator, JensenShannonDivergence
from dmpe.evaluation.model_evaluation import ModelWrapper, ModelEvaluator, PredictionComparison, NodeModelWrapper, EnvWrapper
from dmpe.evaluation.utils import default_constraint_function

from dmpe.utils.env_utils.cart_pole_utils import setup_env as setup_cart_pole_env

In [ ]:
from dmpe.models.model_training import ModelTrainer
from dmpe.evaluation.exp_data_model_learning import train_model_on_experiment_data, ModelExpDataResult

In [ ]:
import optimistix

In [ ]:
env, _ = setup_cart_pole_env()

In [ ]:
env.env_properties.static_params

In [ ]:
def featurize(obs):
    feat_obs = jnp.stack(
        [obs[..., 0], obs[..., 1], jnp.sin(obs[..., 2] * jnp.pi), jnp.cos(obs[..., 2] * jnp.pi), obs[..., 3]],
        axis=-1,
    )
    return feat_obs

In [ ]:
for init_omega in jnp.arange(-1, 1, 0.1):
    init_obs = jnp.array([0, 0, 0., init_omega])
    init_state = env.generate_state_from_observation(init_obs, env.env_properties)
    
    actions = jnp.ones((100,1)) * 0.5
        
    observations, states, last_state = env.sim_ahead(
        init_state,
        actions=actions,
        env_properties=env.env_properties,
        obs_stepsize=env.tau,
        action_stepsize=env.tau,
    )
    plt.plot(states.physical_state.deflection, "b")

plt.hlines(+2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.hlines(-2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.grid()
plt.show()

new_env_properties = env.EnvProperties(
    physical_normalizations=env.env_properties.physical_normalizations,
    action_normalizations=env.env_properties.action_normalizations,
    static_params=env.StaticParams(mu_p=0.002, mu_c=0.5, l=50, m_p=0.1, m_c=1, g=9.81)
)

for init_omega in jnp.arange(-1, 1, 0.1):
    init_obs = jnp.array([0, 0, 0., init_omega])
    init_state = env.generate_state_from_observation(init_obs, env.env_properties)
    
    actions = jnp.ones((100,1)) * 0.5
        
    observations, states, last_state = env.sim_ahead(
        init_state,
        actions=actions,
        env_properties=new_env_properties,
        obs_stepsize=env.tau,
        action_stepsize=env.tau,
    )
    plt.plot(states.physical_state.deflection, "b")

plt.hlines(+2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.hlines(-2.4, xmin=-5, xmax=actions.shape[0] + 5, color="r")
plt.grid()
plt.show()

In [ ]:
observations, states, last_state = env.sim_ahead(
    init_state,
    actions=actions,
    env_properties=new_env_properties,
    obs_stepsize=env.tau,
    action_stepsize=env.tau,
)
env.StaticParams(mu_p=0.002, mu_c=0.5, l=50, m_p=0.1, m_c=1, g=9.81)

In [ ]:
def simulate_ahead_with_env(env, env_properties, init_obs, init_state, actions):
    def body_fun(carry, action):
        obs, state = carry

        obs, state = env.step(state, action, env_properties)
        return (obs, state), obs

    (_, last_state), observations = jax.lax.scan(body_fun, (init_obs, init_state), actions)
    observations = jnp.concatenate([init_obs[None, :], observations], axis=0)

    return observations


def loss_function(static_params, true_observations, batched_actions, env):

    new_env_properties = env.EnvProperties(
        physical_normalizations=env.env_properties.physical_normalizations,
        action_normalizations=env.env_properties.action_normalizations,
        static_params=static_params,
    )
  
    init_obs = true_observations[:, 0]
    init_state = eqx.filter_vmap(env.generate_state_from_observation, in_axes=(0, None))(init_obs, new_env_properties)
    
    # pred_observations, states, last_state = eqx.filter_vmap(env.sim_ahead, in_axes=(0, 0, None, None, None))(
    #     init_state,
    #     batched_actions,
    #     new_env_properties,
    #     env.tau,
    #     env.tau,
    # )
    pred_observations = eqx.filter_vmap(simulate_ahead_with_env, in_axes=(None, None, 0, 0, 0))(
        env,
        new_env_properties,
        init_obs,
        init_state,
        batched_actions
    )
    

    feat_pred_observations = eqx.filter_vmap(featurize)(pred_observations)
    feat_true_observations = eqx.filter_vmap(featurize)(true_observations)

    return jnp.mean((feat_pred_observations - feat_true_observations)**2)

In [ ]:
parameter_gradient = eqx.filter_jit(eqx.filter_grad(loss_function))

In [ ]:
parameter_gradient(
    env.StaticParams(mu_p=jnp.array(0.002), mu_c=jnp.array(0.5), l=jnp.array(0.1), m_p=jnp.array(0.1), m_c=jnp.array(1.), g=jnp.array(9.81)),
    observations[None],
    actions[None],
    env,
)

- no gradient :( why though? **because the inputs where leaves and not jax arrays**
- some operation likely demolishes our ability to differentiate through the env
- differentiating wrt the actions still works, also for the observations, so why not for the static params?

In [ ]:
loss_function(
    env.StaticParams(mu_p=0.002, mu_c=0.5, l=0.5, m_p=0.1, m_c=1, g=9.81),
    observations[None],
    actions[None],
    env,
)

In [ ]:
from dmpe.models.model_training import precompute_starting_points, load_single_batch

In [ ]:
env.env_properties.static_params

In [ ]:
# lol actually load experiment data maybe

In [ ]:
cart_pole_data_path = DataPaths().cs_experiments / "dmpe" / "cart_pole" / "old_results" 
dmpe_experiment_results = load_all_experiment_results(cart_pole_data_path, model_class=NeuralEulerODECartpole)

dmpe_experiment_result = dmpe_experiment_results[0]
n_datapoints = 15_000


exp_id = dmpe_experiment_result["exp_id"]

print(exp_id)
observations = dmpe_experiment_result["observations"][:n_datapoints]
actions = dmpe_experiment_result["actions"][:n_datapoints]
internal_model = dmpe_experiment_result["model"]

In [ ]:
init_guess_params = env.StaticParams(mu_p=0.002, mu_c=0.5, l=jnp.array(0.6), m_p=0.1, m_c=1., g=9.81)
init_guess_params

In [ ]:
n_steps = 5_000
sequence_length = 20
n_train_steps = 1
training_batch_size = 512

estimated_params = deepcopy(init_guess_params)

optimizer = optax.adabelief(1e-3)
opt_state_params = optimizer.init(eqx.filter(estimated_params, eqx.is_inexact_array))

key = jax.random.PRNGKey(0)
key, loader_key = jax.random.split(key, 2)

In [ ]:
for i in tqdm(range(n_steps)):

    starting_points, loader_key = precompute_starting_points(
        n_train_steps=n_train_steps,
        k=observations.shape[0],
        sequence_length=sequence_length,
        training_batch_size=training_batch_size,
        loader_key=loader_key,
    )
    batched_observations, batched_actions = load_single_batch(
        observations, actions, starting_points[0, ...], sequence_length
    )
    
    grads = parameter_gradient(
        estimated_params,
        batched_observations,
        batched_actions,
        env,
    )
    updates, opt_state = optimizer.update(grads, opt_state_params)
    estimated_params = eqx.apply_updates(estimated_params, updates)

In [ ]:
estimated_params

- eval functions
- compare solver to SGD! e.g. through optimistix

In [ ]:
# simulate_ahead_with_env(env, env.env_properties, obs, state, actions)

In [ ]:
starting_points, loader_key = precompute_starting_points(
    n_train_steps=n_train_steps,
    k=observations.shape[0],
    sequence_length=sequence_length,
    training_batch_size=training_batch_size,
    loader_key=loader_key,
)
batched_observations, batched_actions = load_single_batch(
    observations, actions, starting_points[0, ...], sequence_length
)

new_env_properties = env.EnvProperties(
    physical_normalizations=env.env_properties.physical_normalizations,
    action_normalizations=env.env_properties.action_normalizations,
    static_params=estimated_params,
)

init_obs = batched_observations[:, 0]
init_state = eqx.filter_vmap(env.generate_state_from_observation, in_axes=(0, None))(init_obs, new_env_properties)


# pred_observations, states, last_state = eqx.filter_vmap(env.sim_ahead, in_axes=(0, 0, None, None, None))(
#     init_state,
#     batched_actions,
#     new_env_properties,
#     env.tau,
#     env.tau,
# )

pred_observations = eqx.filter_vmap(simulate_ahead_with_env, in_axes=(None, None, 0, 0, 0))(
    env,
    new_env_properties,
    init_obs,
    init_state,
    batched_actions,
)

fig, axs = plt.subplots(1, 5, figsize=(20, 10))

for i in range(4):
    axs[i].plot(pred_observations[0, :, i])
    axs[i].plot(batched_observations[0, :, i], "--")
axs[-1].plot(batched_actions[0, :, i])

print(batched_observations.shape)
print(pred_observations.shape)

plt.show()

for i in range(4):
    plt.plot(batched_observations[0, :20, i] - pred_observations[0, :20, i])
    plt.show()

In [ ]:
starting_points, loader_key = precompute_starting_points(
    n_train_steps=n_train_steps,
    k=observations.shape[0],
    sequence_length=sequence_length,
    training_batch_size=training_batch_size,
    loader_key=loader_key,
)
batched_observations, batched_actions = load_single_batch(
    observations, actions, starting_points[0, ...], sequence_length
)

new_env_properties = env.EnvProperties(
    physical_normalizations=env.env_properties.physical_normalizations,
    action_normalizations=env.env_properties.action_normalizations,
    static_params=env.env_properties.static_params, #estimated_params,
)

init_obs = batched_observations[:, 0]
init_state = eqx.filter_vmap(env.generate_state_from_observation, in_axes=(0, None))(init_obs, new_env_properties)


pred_observations = eqx.filter_vmap(simulate_ahead_with_env, in_axes=(None, None, 0, 0, 0))(
    env,
    new_env_properties,
    init_obs,
    init_state,
    batched_actions,
)

new_env_properties = env.EnvProperties(
    physical_normalizations=env.env_properties.physical_normalizations,
    action_normalizations=env.env_properties.action_normalizations,
    static_params=init_guess_params,
)

init_guess_pred_observations = eqx.filter_vmap(simulate_ahead_with_env, in_axes=(None, None, 0, 0, 0))(
    env,
    new_env_properties,
    init_obs,
    init_state,
    batched_actions,
)


fig, axs = plt.subplots(2, 4, figsize=(16, 10))

for i in range(4):
    axs[0, i].plot(init_guess_pred_observations[0, :, i])
    axs[0, i].plot(batched_observations[0, :, i])

for i in range(4):
    axs[1, i].plot(pred_observations[0, :, i])
    axs[1, i].plot(batched_observations[0, :, i])

In [ ]:
estimated_params

In [ ]:
env.env_properties.static_params

- consider a very small tolerance on the solver?

In [ ]:
init_guess_params = env.StaticParams(mu_p=jnp.array(0.002), mu_c=jnp.array(0.5), l=jnp.array(0.6), m_p=jnp.array(0.1), m_c=jnp.array(1.), g=jnp.array(9.81))
init_guess_params

In [ ]:
n_steps = 5_000
sequence_length = 100
n_train_steps = 1
training_batch_size = 64

estimated_params = deepcopy(init_guess_params)

optimizer = optax.adabelief(1e-3)
opt_state_params = optimizer.init(eqx.filter(estimated_params, eqx.is_inexact_array))

key = jax.random.PRNGKey(0)
key, loader_key = jax.random.split(key, 2)

In [ ]:
def loss_function(actions, state_target):
    state, target = state_target
    
    init_obs = env.generate_observation(state, env.env_properties)
    observations, _ = simulate_ahead_with_env(env, init_obs, state, actions)
    #observations, _, _ = env.sim_ahead(state, actions, env.env_properties, env.tau, env.tau)
    loss = jnp.mean(observations - target[None])**2
    penalties = penalty_function(observations, actions)
    return loss + penalties

In [ ]:
solver = optimistix.BFGS(rtol=1e-10, atol=1e-10)
sol = optimistix.minimise(
    loss_function, solver, proposed_actions, args=(state, target)
)

- huh.. does it not make sense to apply BFGS on a statistical problem? Are there other solvers for it?
- I guess I could optimize it with full gradient computation?